# 09 - Amélioration du modèle

**Objectif** : tenter d'améliorer la forêt aléatoire du notebook 08 (MAE test 518 €/m²) par trois pistes : un autre algorithme (boosting), une cible transformée (log) et la commune comme variable.

**Entrée** : `dvf_77_2025_enrichi.parquet`, même découpage train/test qu'au notebook 08 (`random_state=42`).
**Sortie** : aucun fichier.

**Méthode** : toutes les décisions sont prises par validation croisée sur le train, avec deux validations :
- **aléatoire** (5 parties) : prédire dans des zones connues, un peu optimiste (ventes voisines des deux côtés) ;
- **par commune** (`GroupKFold`) : prédire dans des communes jamais vues, pessimiste pour un usage dans le 77.

Le cas réel est entre les deux : 91,6 % des ventes se font dans des communes de 10 ventes ou plus. Le test, déjà vu une fois au notebook 08, n'est réutilisé qu'à la toute fin.

**Plan**
1. Forêt contre boosting
2. Réglage du boosting
3. Cible en logarithme
4. Commune encodée (target encoding)
5. Évaluation finale sur le test

## 1. Forêt contre boosting

Le boosting (`HistGradientBoostingRegressor`) ajoute de petits arbres l'un après l'autre, chacun corrigeant les erreurs des précédents. `loss="absolute_error"` lui fait minimiser directement la MAE, notre métrique.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GroupKFold
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

df = pd.read_parquet("../data/processed/dvf_77_2025_enrichi.parquet")

variables = ["type_local", "surface_reelle_bati", "nombre_pieces_principales",
             "nb_dependances", "surface_terrain_total", "distance_paris_km",
             "latitude", "longitude"]

df_modele = df.dropna(subset=["distance_paris_km"])
X = df_modele[variables].copy()
X["type_local"] = (X["type_local"] == "Maison").astype(int)
y = df_modele["prix_m2"]

# Même découpage qu'au notebook 08
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
groupes = df_modele.loc[X_train.index, "code_commune"]

def evaluer(modele, nom):
    aleatoire = -cross_val_score(modele, X_train, y_train, cv=5,
                                 scoring="neg_mean_absolute_error").mean()
    par_commune = -cross_val_score(modele, X_train, y_train, cv=GroupKFold(n_splits=5),
                                   groups=groupes, scoring="neg_mean_absolute_error").mean()
    print(f"{nom} | aléatoire : {round(aleatoire)} | par commune : {round(par_commune)}")

evaluer(RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1),
        "Forêt (référence)")
evaluer(HistGradientBoostingRegressor(random_state=42), "Boosting, perte par défaut")
evaluer(HistGradientBoostingRegressor(loss="absolute_error", random_state=42),
        "Boosting, perte MAE")

Forêt (référence) | aléatoire : 513 | par commune : 590
Boosting, perte par défaut | aléatoire : 523 | par commune : 582
Boosting, perte MAE | aléatoire : 517 | par commune : 574


**Lecture** : la forêt de référence redonne bien 513 / 590 (contrôle). Le boosting fait mieux sur des communes jamais vues (574 contre 590) mais pas en aléatoire (517 contre 513). La perte MAE aide un peu par rapport à la perte quadratique par défaut (-6 à -8 €/m²).

## 2. Réglage du boosting

12 combinaisons de `learning_rate`, `max_iter` (nombre d'arbres) et `max_leaf_nodes` (complexité des arbres). `early_stopping=False` pour que toutes les combinaisons soient comparées dans les mêmes conditions (en mode `auto`, il ne s'active qu'au-delà de 10 000 ventes).

In [2]:
def scores(modele):
    aleatoire = -cross_val_score(modele, X_train, y_train, cv=5,
                                 scoring="neg_mean_absolute_error").mean()
    par_commune = -cross_val_score(modele, X_train, y_train, cv=GroupKFold(n_splits=5),
                                   groups=groupes, scoring="neg_mean_absolute_error").mean()
    return round(aleatoire), round(par_commune)

lignes = []
for lr in [0.03, 0.1]:
    for nb_arbres in [100, 300, 600]:
        for feuilles in [15, 31]:
            modele = HistGradientBoostingRegressor(
                loss="absolute_error", learning_rate=lr, max_iter=nb_arbres,
                max_leaf_nodes=feuilles, early_stopping=False, random_state=42
            )
            aleatoire, par_commune = scores(modele)
            lignes.append({"learning_rate": lr, "max_iter": nb_arbres,
                           "max_leaf_nodes": feuilles,
                           "aleatoire": aleatoire, "par_commune": par_commune})

resultats = pd.DataFrame(lignes).sort_values("par_commune")
print(resultats)

    learning_rate  max_iter  max_leaf_nodes  aleatoire  par_commune
4            0.03       600              15        519          571
5            0.03       600              31        511          573
2            0.03       300              15        529          574
7            0.10       100              31        517          574
9            0.10       300              31        510          574
3            0.03       300              31        518          575
11           0.10       600              31        509          575
8            0.10       300              15        515          576
10           0.10       600              15        512          576
6            0.10       100              15        528          577
1            0.03       100              31        544          591
0            0.03       100              15        564          601


**Lecture** : un petit `learning_rate` demande plus d'arbres (0,03 avec 100 arbres : 591-601, le modèle n'a pas fini d'apprendre). Au-delà, 10 combinaisons sur 12 donnent entre 571 et 577 €/m² par commune : c'est un **plateau**, les écarts sont dans le bruit. Retenu : `learning_rate=0.03`, `max_iter=600`, `max_leaf_nodes=31` (511 / 573), bon sur les deux validations. Choisir le meilleur parmi 12 rend son score légèrement optimiste.

## 3. Cible en logarithme

Entraîner sur log(prix) fait raisonner le modèle en erreurs relatives (300 €/m² d'erreur pèse plus sur un bien à 1 500 que sur un bien à 4 500). `TransformedTargetRegressor` remet les prédictions en €/m², donc les scores restent comparables.

In [3]:
from sklearn.compose import TransformedTargetRegressor

hgb = HistGradientBoostingRegressor(
    loss="absolute_error", learning_rate=0.03, max_iter=600,
    max_leaf_nodes=31, early_stopping=False, random_state=42
)
print("Boosting retenu :", scores(hgb))

hgb_log = TransformedTargetRegressor(regressor=hgb, func=np.log, inverse_func=np.exp)
print("Boosting, cible log :", scores(hgb_log))

Boosting retenu : (511, 573)
Boosting, cible log : (510, 571)


**Lecture** : 510 / 571 contre 511 / 573 : aucun gain (bruit). Attendu : avec une perte en erreur absolue, le modèle vise la médiane, que le passage au log ne change pas. **Non retenu** (plus simple sans). À reconsidérer seulement si l'on choisit de mesurer l'erreur en pourcentage.

## 4. Commune encodée (target encoding)

Chaque commune est remplacée par le prix moyen lissé de ses ventes d'entraînement (`TargetEncoder`). Deux protections contre la fuite de données : le calcul croisé interne (une vente n'entre jamais dans son propre encodage) et le `Pipeline` (l'encodeur est réentraîné sur chaque partie, sans voir la validation). Une commune absente de l'entraînement reçoit la moyenne générale.

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder

def scores(modele, X=X_train):
    aleatoire = -cross_val_score(modele, X, y_train, cv=5,
                                 scoring="neg_mean_absolute_error").mean()
    par_commune = -cross_val_score(modele, X, y_train, cv=GroupKFold(n_splits=5),
                                   groups=groupes, scoring="neg_mean_absolute_error").mean()
    return round(aleatoire), round(par_commune)

# X_train + la commune
X_train_c = X_train.copy()
X_train_c["code_commune"] = df_modele.loc[X_train.index, "code_commune"]

encodage = ColumnTransformer(
    [("commune", TargetEncoder(target_type="continuous", random_state=42), ["code_commune"])],
    remainder="passthrough",
)
modele_commune = make_pipeline(encodage, hgb)

print("Boosting sans commune :", scores(hgb))
print("Boosting + commune    :", scores(modele_commune, X_train_c))

Boosting sans commune : (511, 573)


Boosting + commune    : (507, 639)


**Lecture** : en aléatoire, gain de 4 €/m² seulement (bruit) : latitude et longitude localisaient déjà les communes. Par commune, l'erreur **augmente** de 66 €/m² (573 vers 639) : le modèle a appris à se fier à la colonne commune, qui vaut partout la moyenne générale pour des communes inconnues, et tire les prédictions vers la moyenne. **Non retenu** : aucun gain pour les communes connues, une dégradation pour les communes rares ou nouvelles.

## 5. Évaluation finale sur le test

Modèle retenu : boosting (perte MAE, `learning_rate=0.03`, 600 arbres, 31 feuilles), entraîné sur tout le train.

In [6]:
from sklearn.metrics import mean_absolute_error

hgb.fit(X_train, y_train)
pred_hgb = hgb.predict(X_test)
print("MAE test boosting :", round(mean_absolute_error(y_test, pred_hgb)))

MAE test boosting : 516


**Lecture** : 516 €/m² sur le test, cohérent avec la validation croisée (511), contre 518 pour la forêt du notebook 08. Sur ce test (des zones connues), les deux modèles sont équivalents ; l'avantage du boosting n'apparaît que sur des communes jamais vues (573 contre 590 en validation croisée). Le test ayant déjà été vu une fois, ce chiffre est à prendre comme légèrement optimiste.

## Conclusion

| Essai | Aléatoire | Par commune | Retenu |
| --- | --- | --- | --- |
| Forêt (notebook 08) | 513 | 590 | |
| Boosting, perte MAE réglé | 511 | 573 | oui |
| + cible en log | 510 | 571 | non |
| + commune encodée | 507 | 639 | non |

- **Modèle retenu** : boosting, MAE test de 516 €/m² (baseline : 887). Erreur attendue dans le 77 : environ 510 à 575 €/m² selon que la commune est bien connue ou non.
- **Seul gain réel** : le boosting sur des communes jamais vues (-17 €/m²). Réglages, log et commune ne changent rien ou dégradent.
- **Plateau** : le modèle a atteint la limite de l'information disponible. Pour aller plus loin, il faut de nouvelles variables (âge et état du bien, transports, zones commerciales) ou plusieurs années de données, pas un autre algorithme.
- **Limite temporelle** : aucune validation ne mesure le décalage dans le temps ; un modèle entraîné sur 2025 et utilisé en 2026 peut être biaisé si les prix ont évolué.
- **Leçon** : une variable qui paraît utile (la commune) peut dégrader le modèle quand elle se comporte différemment à l'entraînement et à l'usage.